# Dependencies and Stage Boundaries: The Scheduler Algorithm

**Module**: 04_execution_engine  
**Notebook**: 07_dependencies_and_stage_boundaries.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The exact algorithm DAGScheduler uses to traverse dependencies backward and split stages.
- **Why does this topic exist?**: Deep mastery of stage boundary creation enables precise execution prediction.
- **What problem does it solve?**: Unexpected stage creation slowing queries.
- **Where is it used in real-world Data Engineering?**: Senior Data Engineering technical interviews.

---

## 2. The Stage Boundary Algorithm: Traverse backward from terminal RDD, split at ShuffleDependency.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Dependencies_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
rdd = spark.sparkContext.parallelize(range(100), 2).map(lambda x: (x%5, x)).reduceByKey(lambda a, b: a+b)
print("Lineage tree showing ShuffledRDD:")
print(rdd.toDebugString().decode('utf-8'))

---

## Experiment: Verifying Stage Split Algorithm

- **1. What are we testing?**: Check stage count in Spark UI.
- **2. What do we expect?**: 2 stages.

In [ ]:
print('Count result:', rdd.count())

**3. What happened?**: Result computed across 2 stages.  
**4. Why did it happen?**: reduceByKey introduced ShuffleDependency.  
**5. Practical Engineering Lesson**: ShuffleDependency is the sole trigger for stage boundary creation.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Dependencies
**Problem Statement**: Inspect dependencies array.  
**Input / Context**: rdd.dependencies

In [ ]:
# B

### Exercise 2 (Intermediate): Count Shuffles
**Problem Statement**: Count shuffles in complex pipeline.  
**Input / Context**: Provided dataset

In [ ]:
# I

### Exercise 3 (Advanced): Verify in UI
**Problem Statement**: Check UI stages graph.  
**Input / Context**: Provided dataset

In [ ]:
# A

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Dependencies):
```python
print([type(d).__name__ for d in rdd.dependencies])
```

#### Solution 2 (Count Shuffles):
```python
print('Count ShuffleDependencies')
```

#### Solution 3 (Verify in UI):
```python
print('Inspected stage DAG in localhost:4040')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Algorithm**: Backward traversal splitting at ShuffleDependency.
- **Stages**: ShuffleMapStage followed by downstream stage.

### Essential Technical Terms
- `ShuffleDependency`: Triggers stage boundary.
- `OneToOneDependency`: Continues stage.

### Common Engineering Mistakes
- **Thinking each transformation is a stage**: Only wide dependencies create stages.

### Interview Questions on This Topic
- **Q: How does DAGScheduler decide stage boundaries?**
  - *A*: By traversing the RDD lineage backwards from the action and cutting a new stage boundary at every ShuffleDependency.

### What to Study Next
Module 04 complete! Proceed to Module 05: Partitions and Shuffle Internals.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')